# Lección 18: Asegurando agentes de IA con recibos criptográficos

## Cuaderno práctico

Este cuaderno guía a través de cuatro ejercicios:

1. **Firma tu primer recibo** para una llamada a una herramienta de agente y verifícalo.
2. **Manipula el recibo** y observa cómo falla la verificación.
3. **Construye una cadena de tres recibos** y confirma la integridad de la cadena.
4. **Envuelve una llamada a una herramienta del Microsoft Agent Framework** para que cada acción emita un recibo.

Todos los primitivos criptográficos se importan de bibliotecas bien mantenidas (`pynacl` para Ed25519, `jcs` para JSON canónico RFC 8785, `hashlib` de la biblioteca estándar de Python para SHA-256). La lógica del recibo en sí es Python simple que puedes leer y modificar.

Ejecuta las celdas en orden. Cada sección es corta y autocontenida.


## Configuración

Instale las dos dependencias. Ambas tienen licencias permisivas (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Utilidades Auxiliares

Estos dos auxiliares manejan la codificación base64url (sin relleno) y el hash SHA-256 de objetos arbitrarios. Mantienen el resto del cuaderno enfocado en la lógica del recibo en sí.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sección 1: Firma tu primer recibo

Imagina que nuestro agente de **Contoso Travel** acaba de buscar vuelos de Sídney a Los Ángeles para un cliente. Queremos registrar esta llamada a la herramienta como un recibo firmado para que un auditor futuro pueda verificarla sin tener que confiarnos.

### Paso 1.1: Generar una clave de firma

En producción, la clave de firma del agente residiría en un módulo de seguridad de hardware (HSM), Azure Key Vault, o una tienda protegida similar. Para esta lección generamos una clave nueva en memoria.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Paso 1.2: Construir la carga útil del recibo

La carga útil contiene todo lo que queremos que el recibo certifique: quién actuó, qué herramienta, con qué argumentos, qué se devolvió, bajo qué política y cuándo. Hasheamos los argumentos y el resultado en lugar de incluirlos en línea para que el recibo no filtre contenido sensible.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Paso 1.3: Firmar y ensamblar el recibo

Tres pasos:

1. Canonicalizar la carga útil usando JCS para que dos implementaciones que produzcan el mismo recibo lógico generen bytes idénticos.
2. Hashear los bytes canónicos con SHA-256.
3. Firmar el hash con la clave privada Ed25519.

La firma se adjunta luego a la carga útil original para producir el recibo final.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Paso 1.4: Verificar el recibo

La verificación invierte el proceso. Eliminamos la firma, recalculamos el hash canónico y verificamos la firma contra la clave pública en el recibo.

Un auditor que realice esta verificación no necesita nada de nosotros excepto el recibo mismo. No hay servicio que llamar, ni directorio de claves que consultar, ni confianza requerida.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Deberías ver `Receipt is valid: True`. El agente ha producido su primer registro de auditoría firmado criptográficamente.


## Sección 2: Manipular el recibo

Todo el objetivo de los recibos es que sean evidentes de manipulación. Vamos a demostrarlo.

Modificaremos exactamente un carácter del recibo y observaremos que la verificación falla.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ¿Qué acaba de pasar?

Cuando cambiamos `policy_id`, los bytes canónicos cambiaron. El hash SHA-256 de esos bytes cambió. La firma (que estaba sobre el hash original) ya no coincide con el nuevo hash. La verificación correctamente devuelve `False`.

No hay manera de modificar ningún campo del recibo y que aún así se verifique, a menos que el atacante tenga la clave privada. Mientras la clave privada esté en una bóveda de claves y la clave pública esté publicada, es imposible ocultar la manipulación.

Pruébalo tú mismo: modifica el `tool_name` o `agent_id` o `timestamp` en la celda de arriba y vuelve a ejecutar. Cada cambio produce un recibo inválido.


## Sección 3: Encadenar recibos

Un solo recibo protege una acción. La mayoría de los agentes realizan muchas acciones. Para hacer que toda la secuencia sea evidente en caso de manipulación, enlazamos cada recibo con el anterior incluyendo el hash del recibo previo en la carga útil del nuevo recibo.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Si alguien elimina o reordena un recibo, la cadena se rompe en exactamente ese punto. La verificación de cualquier recibo posterior falla porque su `previous_receipt_hash` ya no coincide con el hash real de su predecesor.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Ahora rompe la cadena manipulando el recibo del medio y vuelve a verificar. El recibo manipulado falla en la verificación de su firma, Y el siguiente recibo falla en la verificación del enlace de cadena (porque su `previous_receipt_hash` ya no coincide con el hash del recibo modificado del medio).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

El recibo 0 todavía verifica (no fue modificado y no tiene un predecesor del que dependa). El recibo 1 falla su verificación de firma porque cambiamos `tool_args_hash`. El recibo 2 falla su verificación de enlace en cadena porque su `previous_receipt_hash` se calculó con base en el recibo 1 original (ahora modificado).

Incluso si un atacante vuelve a firmar el recibo 1 modificado (lo cual no puede hacer sin la clave privada), la discrepancia del enlace en cadena en el recibo 2 aún expondría la manipulación. Para ocultar el cambio, el atacante tendría que volver a firmar cada recibo desde el punto de la modificación en adelante, lo que requiere la posesión de la clave privada.


## Sección 4: Envolver una llamada a herramienta de agente con firma de recibo

En una implementación real, no desea que cada autor de agente recuerde llamar a `make_receipt`. Desea que la firma del recibo sea automática para cada invocación de una herramienta.

Aquí está el patrón más simple: una clase envolvente que toma cualquier función herramienta llamable y devuelve una versión que emite recibos. Esto se adapta a cualquier marco de agente, incluido el Microsoft Agent Framework (`agent_framework.foundry`).

Si no tiene un proyecto Microsoft Foundry configurado, el modelo local a continuación todavía demuestra el patrón.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integración con el Microsoft Agent Framework

El envoltorio `ReceiptedTool` anterior es independiente del framework. Para usarlo dentro de un agente construido con el Microsoft Agent Framework, registre la función envuelta como una herramienta. Un boceto (reemplazaría el mock con un registro real de herramienta de Microsoft Foundry):

```python
# Pseudocódigo que muestra la forma de integración.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Eres un agente de viajes de Contoso ...",
#     tools=[receipted_lookup],   # la herramienta envuelta, no la función en sí
# )
# response = agent.run("Encuentra vuelos de Sídney a Los Ángeles en junio.")
#
# # Después de la ejecución, cada llamada a herramienta que hizo el agente tiene un recibo firmado:
# audit_chain = receipted_lookup.receipts
```

El framework del agente no necesita saber nada sobre los recibos. La firma del recibo está envuelta alrededor de la herramienta, no atornillada en el framework. Así es como se añade origen a un código de agente existente sin reescribir el agente.


## Resumen y desafío adicional

Has:

- Generado un par de claves Ed25519.
- Construido y firmado un recibo para una llamada de herramienta de agente.
- Verificado el recibo sin conexión usando solo la clave pública.
- Manipulado un recibo y observado la falla de verificación.
- Construido una secuencia encadenada por hash de tres recibos.
- Manipulado el medio de la cadena y observado tanto la falla de la firma como la falla del enlace en la cadena.
- Envuelto una función de herramienta con firma automática de recibo.

**Desafío adicional.** Extiende el esquema del recibo con un campo `request_id` (un UUID para rastreo distribuido). Actualiza `make_receipt` para incluirlo y confirma que los recibos aún se verifican de extremo a extremo. Luego modifica el campo después de la firma y confirma que la verificación falla. Esto te obliga a internalizar cómo cada byte de la codificación canónica contribuye a la firma.

**Límite importante.** Los recibos prueban tres cosas y solo tres: atribución (esta clave firmó este contenido), integridad (el contenido no ha cambiado desde la firma), y orden (este recibo vino después de ese recibo). NO prueban que la acción del agente fue correcta, que la política nombrada en `policy_id` fue realmente evaluada, o que el agente siguió todas las reglas. Los recibos son una base. La gobernanza es el sistema que construyes encima.

Lee de nuevo el README de la lección con ese límite en mente. El error más común que cometen los equipos con los recibos es asumir que "tenemos recibos" significa "estamos gobernados." No es así. Los recibos hacen que el comportamiento del agente sea auditable. No lo hacen correcto.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
